# Operating-point metrics at FPR <= 0.05 (Table 8)

Precision, recall, F1 at the deployment-relevant operating point for all detectors across four folds.


**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).

**Input dependency.** This notebook is a T4-row extension. It reads the existing 3-fold AUROC values from a CSV produced by an earlier run, computes the T4 row, and merges. Required input:

- `Processed_Data/NB10i_operating_points_fpr05.csv`

If the file is missing, the notebook raises an explicit error.


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_CSV = os.path.join(OUT, "NB10i_operating_points_fpr05.csv")
OUT_CSV       = os.path.join(OUT, "NB10i_operating_points_fpr05_4fold.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Constants
TASKS        = ["T1", "T2", "T3", "T4"]
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0, 0, 0.05])
GRAVITY      = np.array([0, 0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
FPR_TARGET   = 0.05
GMM_MAX_COMP = 8
GMM_MAX_ITER = 500
GMM_N_INIT   = 5

LSTMVAE_EPOCHS   = 80
LSTMVAE_BATCH    = 32
LSTMVAE_LR       = 1e-3
LSTMVAE_HIDDEN   = 64
LSTMVAE_LATENT   = 32
LSTMVAE_N_LAYERS = 2
LSTMVAE_BETA     = 1.0

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3]
                 for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = (Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6 \
                 else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3]
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

# REGISTRY (all 4 folds)
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",       "T1","healthy","none"),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",       "T2","healthy","none"),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",       "T3","healthy","none"),
    "T4_healthy":    ("T4_BinReorient/healthy","UR5_T4_healthy_session2_35cyc_*.h5","T4","healthy","none"),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",     "T4","A2","0.5kg"),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",       "T4","A2","1kg"),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",       "T4","A2","2kg"),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",    "T4","A3","7duct"),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",   "T4","A3","14duct"),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",      "T4","A5","20mm"),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",      "T4","A5","50mm"),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",     "T4","A5","100mm"),
}

PSR_COLS = ([f"J{j}_{s}" for j in range(6)
             for s in ["resid_mean","resid_std","resid_rms","resid_max",
                       "resid_skew","resid_kurtosis",
                       "grav_resid_std","grav_resid_rms"]]
            + ["total_resid_rms","J1J2_resid_corr"])

RAW_COLS = ([f"J{j}_{s}" for j in range(6)
             for s in ["raw_mean","raw_std","raw_rms"]]
            + ["total_raw_rms"])

# Operating-point metric computation
def operating_point_fpr05(y_true, y_score, fpr_target=FPR_TARGET):
    fpr_arr, tpr_arr, thresholds = roc_curve(y_true, y_score)
    feasible = np.where(fpr_arr <= fpr_target)[0]
    if len(feasible) == 0:
        best_idx = np.argmin(fpr_arr[fpr_arr > 0]) if np.any(fpr_arr > 0) else 0
    else:
        best_idx = feasible[np.argmax(tpr_arr[feasible])]
    thresh = float(thresholds[best_idx])
    y_pred = (y_score >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    auroc     = roc_auc_score(y_true, y_score)
    return {
        "threshold": round(thresh, 6),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
        "fpr":       round(fpr, 4),
        "auroc":     round(auroc, 4),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n_healthy": int((y_true == 0).sum()),
        "n_anomaly": int((y_true == 1).sum()),
    }

# PSR fit/extract
def fit_psr(healthy_cycs):
    """Per-joint OLS fit:
    - np.linalg.lstsq (not solve + ridge regularizer)
    - per-cycle TASK_PAYLOAD
    - qdd = 0 at sequence boundaries (not clamped indexing)
    """
    train_Phi = {j: [] for j in range(6)}
    train_I   = {j: [] for j in range(6)}
    for cyc in healthy_cycs:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a = cyc["q"]; qd_a = cyc["qd"]; cur = cyc["current"]
        N = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < N - 1 else 0.0)
                train_Phi[j].append(np.array([tau_g[j], qd_a[t, j],
                                              np.sign(qd_a[t, j]),
                                              qdd_j, 1.0]))
                train_I[j].append(cur[t, j])
    psr_w = {}
    for j in range(6):
        w, _, _, _ = np.linalg.lstsq(
            np.array(train_Phi[j]), np.array(train_I[j]), rcond=None)
        psr_w[j] = w
    # Return as (6, 5) np.array for compatibility with extract_psr_features below
    W = np.zeros((6, 5))
    for j in range(6):
        W[j] = psr_w[j]
    return W

def extract_psr_features(cyc, psr_w):
    payload = TASK_PAYLOAD[cyc["task"]]
    q_a = cyc["q"]; qd_a = cyc["qd"]; cur = cyc["current"]
    N = len(q_a)
    idx = list(range(0, N, SUBSAMPLE))
    res = np.zeros((len(idx), 6)); gr = np.zeros((len(idx), 6))
    for ti, t in enumerate(idx):
        tau_g = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < N - 1 else 0.0)
            phi = np.array([tau_g[j], qd_a[t, j],
                            np.sign(qd_a[t, j]), qdd_j, 1.0])
            res[ti, j] = cur[t, j] - phi @ psr_w[j]
            gr[ti, j]  = cur[t, j] - (psr_w[j][0]*tau_g[j] + psr_w[j][4])
    f = {}
    for j in range(6):
        r = res[:, j]; g = gr[:, j]
        f[f"J{j}_resid_mean"]     = r.mean()
        f[f"J{j}_resid_std"]      = r.std()
        f[f"J{j}_resid_rms"]      = np.sqrt(np.mean(r**2))
        f[f"J{j}_resid_max"]      = np.abs(r).max()
        f[f"J{j}_resid_skew"]     = float(sst.skew(r))
        f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
        f[f"J{j}_grav_resid_std"] = g.std()
        f[f"J{j}_grav_resid_rms"] = np.sqrt(np.mean(g**2))
    f["total_resid_rms"] = np.sqrt(np.mean(res**2))
    f["J1J2_resid_corr"] = float(np.corrcoef(res[:,1], res[:,2])[0,1]
                                  if len(res) > 2 else 0.0)
    return np.array([f[k] for k in PSR_COLS])

def extract_raw_features(cyc):
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    c = cur[idx]
    f = {}
    for j in range(6):
        f[f"J{j}_raw_mean"] = c[:, j].mean()
        f[f"J{j}_raw_std"]  = c[:, j].std()
        f[f"J{j}_raw_rms"]  = np.sqrt(np.mean(c[:, j]**2))
    f["total_raw_rms"] = np.sqrt(np.mean(c**2))
    return np.array([f[k] for k in RAW_COLS])

# LSTM-VAE architecture
class LSTMEncoder(nn.Module):
    def __init__(self, n_channels, hidden_dim, latent_dim, n_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_channels, hidden_size=hidden_dim,
                            num_layers=n_layers, batch_first=True,
                            bidirectional=True)
        self.fc_mu     = nn.Linear(2 * hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(2 * hidden_dim, latent_dim)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h_cat = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc_mu(h_cat), self.fc_logvar(h_cat)

class LSTMDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, n_channels, n_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers
        self.fc_init    = nn.Linear(latent_dim, hidden_dim)
        self.lstm = nn.LSTM(input_size=latent_dim, hidden_size=hidden_dim,
                            num_layers=n_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, n_channels)

    def forward(self, z, seq_len):
        z_rep = z.unsqueeze(1).repeat(1, seq_len, 1)
        h0 = torch.tanh(self.fc_init(z)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        c0 = torch.zeros_like(h0)
        out, _ = self.lstm(z_rep, (h0, c0))
        return self.fc_out(out)

class LSTMVAE(nn.Module):
    def __init__(self, n_channels=6,
                 hidden_dim=LSTMVAE_HIDDEN, latent_dim=LSTMVAE_LATENT,
                 n_layers=LSTMVAE_N_LAYERS, beta=LSTMVAE_BETA):
        super().__init__()
        self.beta    = beta
        self.encoder = LSTMEncoder(n_channels, hidden_dim, latent_dim, n_layers)
        self.decoder = LSTMDecoder(latent_dim, hidden_dim, n_channels, n_layers)

    def reparameterise(self, mu, logvar):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return mu

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z, x.shape[1]), mu, logvar

    def elbo_loss(self, x):
        x_hat, mu, logvar = self.forward(x)
        recon = nn.functional.mse_loss(x_hat, x, reduction="mean")
        kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon + self.beta * kl, recon.item()

    @torch.no_grad()
    def reconstruction_score(self, x):
        self.eval()
        x_hat, _, _ = self.forward(x)
        return ((x - x_hat) ** 2).mean(dim=[1, 2]).cpu().numpy()

# Main
print("=" * 70)
print("NB10i_T4_extension -- T4 operating points at FPR <= 0.05")
print("=" * 70)
print(f"Device: {DEVICE}")

print(f"\n[Step 1] Loading {os.path.basename(CANONICAL_CSV)}...")
if not os.path.exists(CANONICAL_CSV):
    raise RuntimeError(
        f"Canonical operating-points CSV not found at {CANONICAL_CSV}. "
        "This is the source of the published Table 6 T1/T2/T3 values."
    )
canon = pd.read_csv(CANONICAL_CSV)
print(f"  Loaded {len(canon)} rows.")
print(f"  Columns: {list(canon.columns)}")
print(f"  Methods: {sorted(canon['method'].unique())}")

# Step 2: load all cycles
print("\n[Step 2] Loading HDF5 data...")
all_cycles = []
for key, (subdir, pattern, task, anomaly, severity) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING — not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        cur_all = f["actual_current"][:]
        q_all   = f["actual_q"][:]
        qd_all  = f["actual_qd"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "q": q_all[mask], "qd": qd_all[mask],
                "current": cur_all[mask], "task": task,
                "anomaly": anomaly, "severity": severity,
                "is_anomaly": is_anom,
            })
healthy_cycles = [c for c in all_cycles if c["is_anomaly"] == 0]
print(f"  Total cycles: {len(all_cycles)} | Healthy: {len(healthy_cycles)}")
for t in TASKS:
    nh = sum(1 for c in all_cycles if c["task"]==t and c["is_anomaly"]==0)
    na = sum(1 for c in all_cycles if c["task"]==t and c["is_anomaly"]==1)
    print(f"    {t}: {nh} healthy, {na} anomaly")

# Step 3: PSR fit on T1+T2+T3 healthy
print("\n[Step 3] Fitting PSR on T1+T2+T3 healthy...")
t0 = time.perf_counter()
tr_healthy = [c for c in healthy_cycles if c["task"] in ["T1","T2","T3"]]
te_cycles  = [c for c in all_cycles if c["task"] == "T4"]
print(f"  Training healthy: {len(tr_healthy)} cycles")
print(f"  Test cycles (T4): {len(te_cycles)}")
W = fit_psr(tr_healthy)
print(f"  PSR fit: {time.perf_counter()-t0:.1f}s")

# Step 4: extract PSR and Raw features
print("\n[Step 4] Extracting PSR and Raw features...")
t0 = time.perf_counter()
X_psr_te = np.array([extract_psr_features(c, W) for c in te_cycles])
X_psr_tr = np.array([extract_psr_features(c, W) for c in tr_healthy])
X_raw_te = np.array([extract_raw_features(c) for c in te_cycles])
X_raw_tr = np.array([extract_raw_features(c) for c in tr_healthy])
y_te = np.array([c["is_anomaly"] for c in te_cycles])
print(f"  Feature extraction: {time.perf_counter()-t0:.1f}s")
print(f"  PSR features: {X_psr_te.shape}; Raw features: {X_raw_te.shape}")

# Step 5: detector fitting and scoring
print("\n[Step 5] Detector fitting and scoring (T4 fold)...")
detector_scores = {}

# PSR_ZScore
mu_psr = X_psr_tr.mean(0); sg_psr = X_psr_tr.std(0) + 1e-8
detector_scores["PSR_ZScore"] = np.abs((X_psr_te - mu_psr) / sg_psr).mean(1)
print(f"  PSR_ZScore: scored")

# PSR_OCSVM
sc_psr = StandardScaler().fit(X_psr_tr)
clf_svm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
clf_svm.fit(sc_psr.transform(X_psr_tr))
detector_scores["PSR_OCSVM"] = -clf_svm.decision_function(sc_psr.transform(X_psr_te))
print(f"  PSR_OCSVM: scored")

# PSR_IsoForest
# no StandardScaler (uses raw X_psr_tr); no n_jobs (sklearn default = serial)
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_psr_tr)
detector_scores["PSR_IsoForest"] = -iso.decision_function(X_psr_te)
print(f"  PSR_IsoForest: scored")

# GMM (PSR feat)
print(f"  GMM: BIC component selection (1-{GMM_MAX_COMP})...", end=" ")
Xtr_gmm = sc_psr.transform(X_psr_tr); Xte_gmm = sc_psr.transform(X_psr_te)
bic_scores = []
for k in range(1, GMM_MAX_COMP + 1):
    g = GaussianMixture(n_components=k, max_iter=GMM_MAX_ITER,
                        n_init=GMM_N_INIT, random_state=42)
    g.fit(Xtr_gmm)
    bic_scores.append(g.bic(Xtr_gmm))
best_k = np.argmin(bic_scores) + 1
print(f"best_k={best_k}")
gmm = GaussianMixture(n_components=best_k, max_iter=GMM_MAX_ITER,
                      n_init=GMM_N_INIT, random_state=42)
gmm.fit(Xtr_gmm)
detector_scores["GMM"] = -gmm.score_samples(Xte_gmm)
print(f"  GMM: scored")

# Raw_ZScore
mu_raw = X_raw_tr.mean(0); sg_raw = X_raw_tr.std(0) + 1e-8
detector_scores["Raw_ZScore"] = np.abs((X_raw_te - mu_raw) / sg_raw).mean(1)
print(f"  Raw_ZScore: scored")

# LSTM-VAE — needs sequence representation
print(f"  LSTM-VAE: training on T4 fold ({LSTMVAE_EPOCHS} epochs)...")

# Determine FIXED_LEN
sub_lengths = np.array([len(range(0, len(c["current"]), SUBSAMPLE))
                        for c in all_cycles])
p5 = int(np.percentile(sub_lengths, 5))
FIXED_LEN = max(p5 - (p5 % 4), 64)
print(f"    FIXED_LEN={FIXED_LEN}")

def cycle_to_sequence(cyc, fixed_len=FIXED_LEN):
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    ts = cur[idx].astype(np.float32)
    mu = ts.mean(0, keepdims=True); sg = ts.std(0, keepdims=True) + 1e-8
    ts = (ts - mu) / sg
    if len(ts) >= fixed_len:
        return ts[:fixed_len]
    pad = np.zeros((fixed_len - len(ts), ts.shape[1]), dtype=np.float32)
    return np.vstack([ts, pad])

tr_seqs = [cycle_to_sequence(c) for c in tr_healthy]
X_tr_lstm = torch.tensor(np.stack(tr_seqs)).to(DEVICE)
tr_dl = DataLoader(TensorDataset(X_tr_lstm), batch_size=LSTMVAE_BATCH,
                   shuffle=True, drop_last=False)

model = LSTMVAE(n_channels=6).to(DEVICE)
opt = optim.Adam(model.parameters(), lr=LSTMVAE_LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=LSTMVAE_EPOCHS, eta_min=1e-5)

t0 = time.perf_counter()
model.train()
for epoch in range(LSTMVAE_EPOCHS):
    epoch_loss = 0.0
    for (xb,) in tr_dl:
        opt.zero_grad()
        loss, _ = model.elbo_loss(xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        epoch_loss += loss.item() * len(xb)
    scheduler.step()
    if (epoch + 1) % 20 == 0:
        print(f"    Epoch {epoch+1}/{LSTMVAE_EPOCHS}  loss={epoch_loss/len(X_tr_lstm):.6f}")
print(f"    LSTM-VAE training: {time.perf_counter()-t0:.1f}s")

model.eval()
lstm_scores = []
for cyc in te_cycles:
    seq = cycle_to_sequence(cyc)
    x = torch.tensor(seq).unsqueeze(0).to(DEVICE)
    lstm_scores.append(model.reconstruction_score(x)[0])
detector_scores["LSTM-VAE"] = np.array(lstm_scores, dtype=float)
print(f"  LSTM-VAE: scored")

# Step 6: compute operating-point metrics
print("\n[Step 6] Computing FPR<=0.05 operating-point metrics for T4...")
NAME_MAP = {
    "PSR_ZScore":    "PSR_ZScore",
    "PSR_OCSVM":     "PSR_OC-SVM",
    "PSR_IsoForest": "PSR_IsoForest",
    "GMM":           "GMM",
    "Raw_ZScore":    "Raw_ZScore",
    "LSTM-VAE":      "LSTM-VAE",
}

t4_rows = []
for our_name, scores in detector_scores.items():
    canon_name = NAME_MAP[our_name]
    m = operating_point_fpr05(y_te, scores)
    row = {"method": canon_name, "test_task": "T4", **m}
    t4_rows.append(row)
    print(f"  {canon_name:<16}  AUROC={m['auroc']:.4f}  Prec={m['precision']:.4f}  "
          f"Rec={m['recall']:.4f}  F1={m['f1']:.4f}  FPR={m['fpr']:.4f}")

# Step 7: merge and save
print("\n[Step 7] Merging with canonical CSV...")
# Align columns
canon_cols = list(canon.columns)
t4_df = pd.DataFrame(t4_rows)
# Add any missing columns from the 3-fold CSV (filled with NaN)
for c in canon_cols:
    if c not in t4_df.columns:
        t4_df[c] = np.nan
t4_df = t4_df[canon_cols]
combined = pd.concat([canon, t4_df], ignore_index=True)
combined.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV} ({len(combined)} rows)")

# Step 8: integrity check
print("\n[Integrity check] T1/T2/T3 rows preserved byte-for-byte:")
match = True
for _, canon_row in canon.iterrows():
    sel = combined[(combined["test_task"] == canon_row["test_task"]) &
                   (combined["method"] == canon_row["method"])]
    if len(sel) != 1:
        match = False
        print(f"  MISSING: {canon_row['method']} {canon_row['test_task']}")
        continue
    for col in ["auroc", "precision", "recall", "f1", "fpr"]:
        if col in canon_row and col in sel.columns:
            if pd.isna(canon_row[col]) and pd.isna(sel.iloc[0][col]):
                continue
            if abs(float(canon_row[col]) - float(sel.iloc[0][col])) > 1e-6:
                match = False
                print(f"  DRIFT: {canon_row['method']} {canon_row['test_task']} "
                      f"{col}: canon={canon_row[col]} new={sel.iloc[0][col]}")
if match:
    print(f"  All {len(canon)} canonical rows preserved exactly.")

# Step 9: print the assembled 4-fold table.
print("\n" + "=" * 125)
print("REVISED TABLE 6 (Operating-point metrics at FPR<=0.05, 4-fold LOTO)")
print("=" * 125)
TABLE6_METHODS = [
    ("PSR Z-Score",     "PSR_ZScore"),
    ("PSR IsoForest",   "PSR_IsoForest"),
    ("PSR OC-SVM",      "PSR_OC-SVM"),
    ("GMM (PSR feat.)", "GMM"),
    ("LSTM-VAE",        "LSTM-VAE"),
    ("Raw Z-Score",     "Raw_ZScore"),
]

hdr = f"  {'Method':<18}"
for t in TASKS:
    hdr += f"{t+' AUROC':<10}{t+' Prec':<10}{t+' Rec':<10}{t+' F1':<10}{t+' FPR':<10}"
print(hdr)
print("  " + "-" * 122)

for disp_name, csv_name in TABLE6_METHODS:
    row = f"  {disp_name:<18}"
    for task in TASKS:
        sub = combined[(combined["method"] == csv_name) &
                       (combined["test_task"] == task)]
        if len(sub) == 0:
            row += f"{'—':<10}" * 5
            continue
        r = sub.iloc[0]
        row += (f"{r['auroc']:<10.3f}{r['precision']:<10.3f}"
                f"{r['recall']:<10.3f}{r['f1']:<10.3f}{r['fpr']:<10.3f}")
    print(row)
print("=" * 125)

print("\nNB10i_T4_extension complete.")
print("Note: Conv-AE excluded from Table 6 (T1 + T4 inversions make FPR<=0.05 meaningless).")